# Composite Gate Full Backtest (k=10)

Datastream US-only Phase 2 backtests for P-gated U ranking:

- long pool: P > 0.5 + threshold
- short pool: P < 0.5 - threshold
- rank within the gated pools by U

This notebook reports full-sample and per-period pre/post-cost returns and Sharpe ratios. It also tracks:

- number of trading days with both long and short positions
- mean long/short positions per day


In [4]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str((Path('..') / 'src').resolve()))

from krauss.evaluation.phase2_ds_backtest_utils import (
    FAMILY_LABELS,
    FAMILIES,
    K,
    gated_activity_stats,
    load_phase2_ds_data,
    run_gated_backtest,
    sharpe,
    summary_stats,
)

pred, returns = load_phase2_ds_data()
THRESHOLDS = [0.02, 0.03, 0.05]


In [5]:
summary_rows = []
period_rows = []

for family in FAMILIES:
    for threshold in THRESHOLDS:
        _, _, daily = run_gated_backtest(
            pred,
            returns,
            p_col=f'p_{family}',
            u_col=f'u_{family}',
            k=K,
            threshold=threshold,
        )
        stats = summary_stats(daily)
        activity = gated_activity_stats(daily)
        summary_rows.append({
            'Model': FAMILY_LABELS[family],
            'Threshold': threshold,
            **stats,
            **activity,
        })

        if not daily.empty:
            for pid, grp in daily.groupby('period_id', sort=True):
                active = grp[(grp['n_long'] > 0) & (grp['n_short'] > 0)]
                pre = grp['port_ret']
                post = grp['port_ret_net']
                period_rows.append({
                    'Model': FAMILY_LABELS[family],
                    'Threshold': threshold,
                    'period_id': int(pid),
                    'start': grp['date'].min().date(),
                    'end': grp['date'].max().date(),
                    'days': int(len(grp)),
                    'trading_days': int(len(active)),
                    'mean_long': grp['n_long'].mean(),
                    'mean_short': grp['n_short'].mean(),
                    'daily_pre': pre.mean(),
                    'daily_post': post.mean(),
                    'ann_pre': pre.mean() * 252,
                    'ann_post': post.mean() * 252,
                    'sharpe_pre': sharpe(pre),
                    'sharpe_post': sharpe(post),
                })

summary = pd.DataFrame(summary_rows).sort_values(['Model', 'Threshold']).set_index(['Model', 'Threshold'])
summary[['Daily Pre', 'Daily Post', 'Ann Pre', 'Ann Post']] = summary[['Daily Pre', 'Daily Post', 'Ann Pre', 'Ann Post']] * 100
display(summary.style.format({
    'Daily Pre': '{:.4f}%',
    'Daily Post': '{:.4f}%',
    'Ann Pre': '{:.2f}%',
    'Ann Post': '{:.2f}%',
    'Sharpe Pre': '{:.3f}',
    'Sharpe Post': '{:.3f}',
    'Days': '{:.0f}',
    'Trading Days': '{:.0f}',
    'Mean Long/Day': '{:.2f}',
    'Mean Short/Day': '{:.2f}',
    'Mean Long/Active': '{:.2f}',
    'Mean Short/Active': '{:.2f}',
}).set_caption('P-gated U ranking k=10 full-sample summary'))

per_period = pd.DataFrame(period_rows)
display(per_period.style.format({
    'mean_long': '{:.2f}',
    'mean_short': '{:.2f}',
    'daily_pre': '{:.5f}',
    'daily_post': '{:.5f}',
    'ann_pre': '{:.3f}',
    'ann_post': '{:.3f}',
    'sharpe_pre': '{:.3f}',
    'sharpe_post': '{:.3f}',
}).set_caption('P-gated U ranking k=10 per-period statistics'))


,Model,Threshold,period_id,start,end,days,trading_days,mean_long,mean_short,daily_pre,daily_post,ann_pre,ann_post,sharpe_pre,sharpe_post
0,DNN,0.020000,28,2020-10-05,2021-09-29,185,185,8.64,6.15,-0.00024,-0.00067,-0.061,-0.170,-0.117,-0.326
1,DNN,0.020000,29,2022-01-21,2022-04-26,12,12,1.00,4.42,-0.00965,-0.00985,-2.432,-2.483,-2.057,-2.100
2,DNN,0.030000,28,2020-10-06,2021-09-29,77,77,2.70,4.81,-0.00352,-0.00374,-0.888,-0.942,-1.338,-1.420
3,DNN,0.050000,28,2021-03-08,2021-03-08,1,1,1.00,1.00,0.25946,0.25941,65.384,65.371,nan,nan
4,XGB,0.020000,23,2015-10-16,2016-10-12,250,250,9.43,10.00,-0.00001,-0.00137,-0.002,-0.345,-0.006,-1.146
5,XGB,0.020000,24,2016-10-13,2017-10-10,250,250,9.98,9.98,0.00005,-0.00135,0.014,-0.341,0.090,-2.224
6,XGB,0.020000,25,2017-10-11,2018-10-08,250,250,9.98,10.00,0.00074,-0.00077,0.186,-0.194,1.075,-1.121
7,XGB,0.020000,26,2018-10-09,2019-10-07,250,250,9.74,10.00,0.00188,0.00030,0.473,0.075,2.269,0.360
8,XGB,0.020000,27,2019-10-08,2020-10-02,250,250,9.96,10.00,-0.00043,-0.00200,-0.107,-0.504,-0.318,-1.495
9,XGB,0.020000,28,2020-10-05,2021-09-30,250,250,9.98,9.80,0.00005,-0.00127,0.012,-0.319,0.034,-0.888
